<a href="https://colab.research.google.com/github/bcarmoai/ML-and-AI/blob/main/RAG_Chapter2_Demo2_VectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 145.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]


In [ ]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# Directory to store downloaded and extracted data
DATA_DIR = Path("./mimic_textbooks")

# Download and extract the dataset zip file
def download_and_extract_zip(url, extract_to=DATA_DIR):
    # Ensure the directory exists
    extract_to.mkdir(parents=True, exist_ok=True)

    # Download the zip file
    zip_path = extract_to / "textbooks.zip"
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)

    # Extract the zip file
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset downloaded and extracted.")

# Download and extract textbooks
dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
download_and_extract_zip(dataset_url)


88121KB [00:04, 19192.12KB/s]


Extracting dataset...
Dataset downloaded and extracted.


In [ ]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob

# Load text files
def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

# Cleaning and preprocessing function
def clean_and_tokenize(text):
    # Basic regex cleaning
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Lowercase all text
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters

    # Tokenize with gensim
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

# Spell correction
def correct_spelling(text):
    return str(TextBlob(text).correct())

# Chunk text into fixed-size chunks
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, correct, and chunk documents
documents = load_text_files(DATA_DIR / "textbooks/en")
cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
# corrected_documents = [correct_spelling(doc) for doc in cleaned_documents]
chunked_documents = []
for doc in cleaned_documents:
    chunked_documents.extend(chunk_text(doc))

print(f"Total document chunks created: {len(chunked_documents)}")


Total document chunks created: 60061


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# The SentenceTransformer class automatically uses the PyTorch backend (which is stable in Colab)
# and detects the GPU if available.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for all document chunks in batches.
# The encode() method automatically handles tokenization, batching, and mean pooling.
print("Generating embeddings... This might take a moment.")
embeddings = model.encode(chunked_documents, batch_size=128, show_progress_bar=True)
print(f"Generated embeddings for {len(embeddings)} document chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings... This might take a moment.


Batches:   0%|          | 0/470 [00:00<?, ?it/s]

Generated embeddings for 60061 document chunks.


Recent versions of TensorFlow (2.16 and newer) switched to Keras 3 by default, which breaks compatibility with the transformers library's TensorFlow backend unless a compatibility package called tf-keras is installed. I will generate a cell to check your environment for this exact issue.

Based on the diagnostic output, we can see exactly what the incompatibility is: your environment is running a very recent version of TensorFlow (which uses Keras 3 by default, as seen by the Keras 3.x version).

The transformers library's TensorFlow backend is currently built around Keras 2. Even though tf_keras (the legacy Keras 2 compatibility package) is installed, transformers will silently fail to initialize its TensorFlow classes (like TFAutoModel) unless you explicitly tell it to use the legacy Keras before importing anything.

If you ever need to use the TensorFlow backend of transformers in the future, you must set this environment variable before any imports:

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

However, since we already successfully switched your code to use the PyTorch backend via sentence-transformers (which is highly optimized for this), you don't need to worry about this TensorFlow/Keras version conflict for your current pipeline!

The error NameError: name 'index' is not defined occurred because the cell that builds the FAISS index was skipped or not executed in order.

To fix this, please run the cell directly above it (which contains index = faiss.IndexFlatL2(dimension)) to create and populate the index with your document embeddings. Once that cell finishes, you can re-run the query cell and it should successfully retrieve the most similar document chunks!

To summarize our findings and fixes:

The TensorFlow Incompatibility: The issue with TFAutoModel was caused by your environment using TensorFlow 2.16+ which defaults to Keras 3. The transformers library requires Keras 2 (tf_keras). To make them compatible in the future, you must set os.environ["TF_USE_LEGACY_KERAS"] = "1" before importing transformers. However, switching to sentence-transformers cleanly bypassed this issue and provided a more optimized PyTorch-based solution.

The FAISS Index Error: The NameError: name 'index' is not defined occurred simply because the cell responsible for creating the FAISS index (the one containing index = faiss.IndexFlatL2(dimension)) was skipped. Running that cell before your query cell resolves the issue.

In [ ]:
import transformers
import tensorflow as tf

print(f"TensorFlow installed version: {tf.__version__}")
print(f"Transformers version: {transformers.__version__}")

try:
    from transformers import TFAutoModel
    print("\nTFAutoModel is available in transformers.")
except ImportError as e:
    print(f"\nFailed to import TFAutoModel: {e}")
    print("This indicates that the transformers library is not configured to expose TensorFlow classes in this environment, possibly due to version incompatibilities between transformers, tensorflow, and keras.")

TensorFlow installed version: 2.21.0
Transformers version: 5.20.2

Failed to import TFAutoModel: cannot import name 'TFAutoModel' from 'transformers' (/usr/local/lib/python3.13/dist-packages/transformers/__init__.py)
This indicates that the transformers library is not configured to expose TensorFlow classes in this environment, possibly due to version incompatibilities between transformers, tensorflow, and keras.


In [ ]:
import sys
import tensorflow as tf

print("--- Transformers TensorFlow Backend Diagnostics ---")
print(f"TensorFlow version: {tf.__version__}")

try:
    import keras
    print(f"Keras version: {keras.__version__}")
except ImportError:
    print("Keras is not installed.")

try:
    import tf_keras
    print(f"tf_keras version: {tf_keras.__version__}")
except ImportError:
    print("tf_keras is NOT installed.")
    print("\nVERDICT: If TensorFlow >= 2.16, transformers requires 'tf_keras' to be installed. Missing 'tf_keras' is likely why TFAutoModel failed to load.")

# Let's also check if transformers logs any specific error internally
import logging
import transformers
logging.getLogger("transformers").setLevel(logging.DEBUG)

try:
    from transformers.utils import is_tf_available
    print(f"\nTransformers internal TF check (is_tf_available): {is_tf_available()}")
except ImportError:
    try:
        from transformers.utils.import_utils import is_tf_available
        print(f"\nTransformers internal TF check (is_tf_available): {is_tf_available()}")
    except Exception as e:
        print(f"\nCould not run internal check: {e}")


--- Transformers TensorFlow Backend Diagnostics ---
TensorFlow version: 2.21.0
Keras version: 3.13.2
tf_keras version: 2.21.0

Could not run internal check: cannot import name 'is_tf_available' from 'transformers.utils.import_utils' (/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py)


In [ ]:
import faiss
import numpy as np

# Define the dimension of embeddings
dimension = 384  # Embedding size from MiniLM model
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to NumPy array for FAISS
embedding_matrix = np.array([embedding.flatten() for embedding in embeddings]).astype('float32')

# Add embeddings to FAISS index
index.add(embedding_matrix)
print(f"Total embeddings indexed: {index.ntotal}")


Total embeddings indexed: 60061


In [ ]:
import numpy as np

# Example query for testing
def get_embedding(text):
    # model.encode() directly returns the numpy array for the text
    return model.encode(text)

query_text = "What are causes of heart failure?"
query_embedding = get_embedding(query_text)
query_embedding = np.array(query_embedding).reshape(1, -1).astype('float32')

# Search FAISS for the most similar documents
k = 5  # Number of closest documents to retrieve
distances, indices = index.search(query_embedding, k)

# Retrieve and print the most similar chunks
print("Top similar document chunks:")
for idx in indices[0]:
    print("-", chunked_documents[idx])
    print()

Top similar document chunks:
- down to six principal mechanisms failure of the pump in the most common situation the cardiac muscle contracts weakly and the chambers cannot empty systolic dysfunction in some cases the muscle cannot relax sufficiently to permit ventricular filling resulting in diastolic dysfunction obstruction to flow lesions that prevent valve opening eg calcific aortic valve stenosis or cause increased ventricular chamber pressures eg systemic hypertension or aortic coarctation can overwork the myocardium which has to pump against the obstruction regurgitant flow valve pathology that allows backward flow of blood results in increased volume workload and may overwhelm the pumping capacity of the affected chambers shunted flow defects congenital or acquired that divert blood inappropriately from one chamber to another or from one vessel to another lead to pressure and volume overloads disorders of cardiac conduction uncoordinated cardiac impulses or blocked conduction p